(sec:environment)=
# Environment

VeloxChem implements both implicit (CPCM and SMD) and explicit (polarizable embedding) solvation models. 

## Implicit solvation

Implicit solvation models describe the effect of a surrounding liquid environment by replacing the explicit solvent molecules with a continuous dielectric medium that interacts self‑consistently with the electronic structure of the solute. {cite}`Tomasi2005`

A separation is made between equilibrium and non-equilibrium solvation. In the former case, the timescale is such that both nuclear and electronic relaxations take place in the environment, such as in molecular structure optimizations. In the latter case, only electrons are fully equilibrated with the time-dependent solute charge density, such as in UV/vis spectrum simulations. 

(sec:cpcm)=
### CPCM

In the conductor‑like polarizable continuum model (CPCM), the solute is placed inside a cavity defined by its molecular surface, and the reaction field is obtained by solving surface‑charge equations that approximate the dielectric screening of a perfect conductor and are subsequently scaled to represent the desired solvent permittivity.

VeloxChem implements the CPCM model for:

- SCF energies
- gradients (structure optimizations)
- linear response (UV/vis spectra and more)

**Python script**

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("ammonia")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.solvation_model = "cpcm"
scf_results = scf_drv.compute(molecule, basis)

rsp_drv = vlx.LinearResponseEigenSolver()
rsp_drv.nstates = 10
rsp_results = rsp_drv.compute(molecule, basis, scf_results)

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_results = opt_drv.compute(molecule, basis, scf_results)

:::{note}
Water is the default solvent. For other solvents, set the dielectric constant:
:::{code}
scf_drv.cpcm_epsilon = 24.5  # ethanol
:::
:::

**Text file**

:::{code}
@jobs
task: scf
@end

@method settings
basis: def2-svp
xcfun: b3lyp
solvation model: cpcm
cpcm epsilon : 78.39
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
:::

(sec:smd)=
### SMD

The solvation model based on density (SMD) combines a self‑consistent reaction‑field description of the electrostatic polarization with empirically parametrized terms for cavitation, dispersion, and solvent–solute interactions based on the solute’s electron density, enabling accurate free‑energy predictions across a wide range of solvents. For further details, see {cite}`Marenich2009`.

In Veloxchem, the electrostatic contribution is determined using the CPCM model.

**Python script**

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("methanol")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()

scf_drv.solvation_model = "smd"
scf_drv.smd_solvent = "water"

scf_results = scf_drv.compute(molecule, basis)

**Text file**

:::{code}
@jobs
task: scf
@end

@method settings
basis: def2-svp
xcfun: b3lyp
solvation model: smd
smd solvent : water
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
:::

(sec:pe)=
## Polarizable and electrostatic embedding

An explicit representation of the environment is available with:

- Polarizable embedding (PE), where molecules in the environment are represented by site charges and polarizabilities.

- Non-polarizable (NPE), where the environment is represented by point charges only.

**Python script**

Either a single PDB, containing one or more structures, or a trajectory can be passed to the `EnsembleParser`. In this step, the PE and NPE cutoffs are selected:

In [ ]:
import veloxchem as vlx

ens_parser = vlx.EnsembleParser()

ensemble = ens_parser.structures(
    trajectory_file = "../input_files/alpha-helix-acetone-water.xtc",
    topology_file = "../input_files/alpha-helix-acetone-water.tpr",
    qm_region = 'resname LIG',
    num_snapshots = 2,
    pe_cutoff = 3.0,
    npe_cutoff = 5.0,
)

When providing a time-resolved trajectory, for example, `.xtc` together with `.tpr`, several options can be specified:
- `num_snapshots`: Controls snapshot extraction.
    - _Default_: All snapshots used.
    - _If provided_: Snapshots are selected evenly spaced.
- `start`: Start time of the trajectory window in ps. _Default_: Time of the first snapshot.
- `end`: End time of the trajectory window in ps. _Default_: Time of the last snapshot.
- `last_snapshot_only = True`: Processes only the final snapshot.

**Key Parameters**:
- `qm_region`: MDAnalysis selection string that defines QM region.
- `qm_charge`: charge of the QM region
- `env_region`: MDAnalysis selection string that defines environment. If `env_region` is not defined, the entire system is treated as QM.
    - _Default_: All atoms not included in qm_region.

The number of residues treated with PE and NPE can be accessed as follows:

In [7]:
print("number residues PE = ", ensemble[0]["number_residues_pe"])
print("number residues NPE = ", ensemble[0]["number_residues_npe"])

number residues PE =  9
number residues NPE =  14


#### Polarizable and electrostatic  models

Once the environment has been defined, the environment models can be selected with the `set_env_models` method of the `EnsembleDriver`:

In [ ]:
ens_drv = vlx.EnsembleDriver()

ens_drv.set_env_models(
    pe_model=["CP3", "SEP"],
    npe_model=["ff19sb", "tip3p"],
)

VeloxChem supports the following environment models for PE and NPE:
- PE: `CP3` {cite}`Reinholdt2020` for proteins, and `SEP` {cite}`Beerepoot2016` for common polar and non-polar solvent molecules and ions.
- NPE: `ff19sb` {cite}`Tian2020` for proteins, and `tip3p` {cite}`Jorgensen1983` for water.

:::{image} ../images/models-no-cite.png
:align: center
:width: 600px
:::

#### Automatic SCF and spectrum calculations

The SCF calculations can now be automated by passing `scf_options` and, optionally, `property_options` to the `compute` method:

In [ ]:
scf_options = {
   "scf_type": "restricted",
   "conv_thresh": 1.0e-6,
   "max_iter": 150,
   "xcfun": "CAM-B3LYP",
   "grid_level": 4,
}

property_options = {
    "property": "absorption",
    "nstates": 3,
    "nto": True,
}

In [ ]:
results = ens_drv.compute(
   ensemble,
   basis_set = "6-31G",
   scf_options = scf_options,
   property_options = property_options,
)

In some situations, for example when preparing input files for running on a cluster, it is convenient to use the `write_pot_files` method to generate the potential files:


In [ ]:
ens_drv.write_pot_files(ensemble)

Finally, the averaged spectra can be plotted:

In [ ]:
ens_drv.plot_uv_vis_spectra(
    results,
    show_individual = True,
    show_sticks = True,
    xlim_nm = (120,200)
)

:::{image} ../images/averaged_spectra.png
:align: center
:width: 600px
:::

**Text file**

:::{code}
@jobs
task: scf
@end

@method settings
basis: aug-cc-pvdz
potfile: pe_frame_000000.pot
@end

@molecule
charge: 0
multiplicity: 1
xyz:
H   23.09239578 21.92239571 22.43439484
C   23.3823967  22.95239449 22.65439606
H   24.33239555 23.14239693 22.14439583
H   22.6023941  23.562397   22.18439484
C   23.44239616 23.26239395 24.09439468
O   23.01239586 22.44239616 24.92439651
C   24.08239555 24.58239555 24.50439644
H   24.36239624 24.5623951  25.55439568
H   24.92239571 24.80239487 23.84439659
H   23.46239662 25.46239471 24.35439491
@end
:::


The potential file named `pe_frame_000000.pot` in this example takes the following form: [`pe_frame_000000.pot`](../input_files/pe_frame_000000.pot)